# EyeSight AI · Entrenamiento del modelo (Google Colab)

Este cuaderno ajusta **YOLOv8n** con el conjunto de datos ampliado, contrasta la hipótesis del numeral 2.3 (caso de prueba **CP-01**) y exporta el modelo a **LiteRT con cuantización INT8** para la aplicación.

**Antes de empezar:** menú *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4* (gratis).

Para descargar de Roboflow guarda tu clave en el ícono de la llave (🔑 *Secrets*) con el nombre `ROBOFLOW_API_KEY`. La clave **nunca** se escribe en el código ni se sube a GitHub.

## 1. Código del proyecto y dependencias

In [ ]:
!git clone https://github.com/dramirezdlp99/eyesight-ai.git
%cd eyesight-ai/ml
!pip install -q -r requirements.txt roboflow
!python -m pytest -q

## 2. Conjuntos de datos

Busca en [Roboflow Universe](https://universe.roboflow.com/) conjuntos con las clases nuevas (hueco/*pothole*, bolardo/*bollard*, poste/*pole*, escalón o bordillo/*stairs*, *curb*, cono/*traffic cone*, obra/*construction barrier*) y también con personas y vehículos. En cada conjunto: **Download Dataset → YOLOv8 → show download code** y copia `workspace`, `project` y `version` en la lista de abajo.

Tus propias fotos de Pasto: súbelas a un proyecto tuyo de Roboflow, etiquétalas con los nombres de las clases y agrégalo igual a la lista.

In [ ]:
from google.colab import userdata
from roboflow import Roboflow

# (workspace, proyecto, versión) — complétalo con los conjuntos que elijas.
FUENTES = [
    # ("mi-workspace", "pasto-obstaculos", 1),
]

rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
for ws, project, version in FUENTES:
    rf.workspace(ws).project(project).version(version).download("yolov8", location=f"datasets/raw/{project}")

## 3. Unir las fuentes con las clases de EyeSight AI

In [ ]:
!python scripts/prepare_dataset.py --sources datasets/raw/* --output datasets/eyesight

## 4. Ajuste fino de YOLOv8n (aprendizaje por transferencia)

In [ ]:
!python scripts/train.py --data datasets/eyesight/eyesight.yaml --epochs 100 --imgsz 320

## 5. CP-01: modelo preentrenado frente a modelo ajustado (hipótesis)

In [ ]:
!python scripts/evaluate.py --data datasets/eyesight/eyesight.yaml \
    --tuned runs/eyesight/yolov8n_ajustado/weights/best.pt --out results

## 6. Exportar a LiteRT INT8 e instalar en `assets/models`

In [ ]:
!python scripts/export.py --weights runs/eyesight/yolov8n_ajustado/weights/best.pt \
    --data datasets/eyesight/eyesight.yaml
# Evalúa también el archivo INT8 que usará la app (pérdida por cuantización).
!python scripts/evaluate.py --data datasets/eyesight/eyesight.yaml \
    --tuned ../assets/models/eyesight_yolov8n.tflite --out results_int8

## 7. Descargar los resultados

In [ ]:
!zip -qr /content/eyesight_modelo.zip ../assets/models results results_int8 runs/eyesight/yolov8n_ajustado/*.png runs/eyesight/yolov8n_ajustado/*.csv
from google.colab import files
files.download("/content/eyesight_modelo.zip")